In [ ]:
# Import required libraries

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# Load .env file and check for API_KEY exists

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Groq, we can use the OpenAI python client
# And OpenAI allows you to change the base_url

groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"

groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
# Sample code
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

### Inference scaling

In [ ]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

In [ ]:
# We use reasoning effort to increafe the efficiency of the model
# gpt is cheaper and faster model when reasoning effort is low gives right answer whereas reasoning effort min in above cell gives wrong answer
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

In [ ]:
# Using a smarter and slower model with reasoning effort minimal gives a right answer
# This is inference
response = openai.chat.completions.create(model="gpt-5-mini", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

In [ ]:
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [ ]:
gpt_model = "gpt-4.1-mini"
ollama_model = "deepseek-r1:1.5b"
groq_model = "llama-3.3-70b-versatile"

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

ollama_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

groq_system = "You are an sarcastic, and comedy chatbot. You like taunting gpt_system \
    You dont taunt with ollama as he is polite but like to play with gpt. "

gpt_messages = ["Hi there"]
ollama_messages = ["Hi"]
groq_messages = ["How are you guys"]

In [ ]:
from itertools import zip_longest


def call_gpt_OG():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, ollama, groq in zip_longest(gpt_messages, ollama_messages, groq_messages):
        if gpt is not None:
            messages.append({"role": "assistant", "content": f"gpt said: {gpt}"})
        if ollama is not None:
            messages.append({"role": "assistant", "content": f"ollama said: {ollama}"})
        if groq is not None:
            messages.append({"role": "assistant", "content": f"groq said: {groq}"})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_gpt_OG()

### Now lets build chatbots using three LLM models gpt, ollama and groq

In [ ]:
# chat completion request for gpt
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, ollama in zip(gpt_messages, ollama_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": ollama})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
# chat completion request for ollama

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

def call_ollama():
    messages = [{"role": "system", "content": ollama_system}]

    for gpt, ollama in zip(gpt_messages, ollama_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": ollama})

    if gpt_messages:
        messages.append({"role": "user", "content": gpt_messages[-1]})

    response = ollama_client.chat.completions.create(
        model="deepseek-r1:1.5b",
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
# chat completion request for groq

groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = groq_api_key
)

def call_groq():
    messages = [{"role": "system", "content": groq_system}]

    for ollama, groq in zip(ollama_messages, groq_messages):
        messages.append({"role": "user", "content": ollama})
        messages.append({"role": "assistant", "content": groq})

    if ollama_messages:
        messages.append({"role": "user", "content": ollama_messages[-1]})

    response = groq_client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
# Building conversation pipleline for between three LLM's

gpt_messages = ["Hi there"]
ollama_messages = ["Hi"]
groq_messages = ["How are you"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Ollama:\n{ollama_messages[0]}\n"))
display(Markdown(f"### Groq: \n{groq_messages[0]}\n"))

for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"###  Gpt:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)

    ollama_next = call_ollama()
    display(Markdown(f"### Ollama: \n{ollama_next}\n"))
    ollama_messages.append(ollama_next)

    groq_next = call_groq()
    display(Markdown(f"### Groq: \n{groq_next}\n"))
    groq_messages.append(groq_next)

In [ ]:
# Now creating a stream UI Interface using gradio
# Let's create a call that streams back results

import gradio as gr

system_message = "You are a helpful assistant that responds in markdown without code blocks"

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a lazyperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
# Using groq to stream 

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

def stream_groq(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            result += chunk.choices[0].delta.content
            yield result

message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for Groq (e.g., Llama 3)",
    lines=7
)

# Streaming works best into a Textbox
message_output = gr.Textbox(label="Response:", lines=15)

view = gr.Interface(
    fn=stream_groq,
    title="Groq Mini",
    inputs=message_input,          # single input can be passed directly
    outputs=message_output,        # single output can be passed directly
    examples=[
        ["Explain the Transformer architecture to a lazyperson"],
        ["Explain the Transformer architecture to an aspiring AI engineer"],
    ],
    flagging_mode="never",
)

view.launch()

In [23]:
# Using stream model to choose your favourite LLM

def stream_model(prompt, model):
    try:
        model = model.strip().lower()

        if model == "gpt":
            yield from stream_gpt(prompt)
        elif model == "groq":
            yield from stream_groq(prompt)
        else:
            yield f"Error: Unknown model '{model}'"

    except Exception as e:
        yield f"Error: {str(e)}"

message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for the LLM",
    lines=7
)

model_selector = gr.Dropdown(
    choices=["gpt", "groq"],
    label="Select model",
    value="gpt"
)

message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs",
    inputs=[message_input, model_selector],
    outputs=message_output,
    examples=[
        ["Explain the Transformer architecture to a layperson", "gpt"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "groq"]
    ],
    flagging_mode="never"
)

view.launch()

Keyboard interruption in main thread... closing server.
